$\newcommand{\ket}[1]{\left\lvert #1\right\rangle}
\newcommand{\bra}[1]{\left\langle #1\right\rvert}
\newcommand{\braket}[1]{\left\langle #1\right\rangle}
\newcommand{\proj}[1]{\lvert #1\rangle\!\langle #1\rvert}
\newcommand{\tr}{\operatorname{tr}}\newcommand{\Tr}{\operatorname{Tr}}
\newcommand{\Var}{\operatorname{Var}}\newcommand{\Cov}{\operatorname{Cov}}
\newcommand{\rank}{\operatorname{rank}}\newcommand{\diag}{\operatorname{diag}}
\newcommand{\poly}{\operatorname{poly}}\newcommand{\polylog}{\operatorname{polylog}}
\newcommand{\norm}[1]{\lVert #1\rVert}\newcommand{\abs}[1]{\lvert #1\rvert}
\newcommand{\bigo}{\mathcal{O}}\newcommand{\id}{\mathbb{I}}\newcommand{\dg}{^{\dagger}}
\newcommand{\R}{\mathbb{R}}\newcommand{\C}{\mathbb{C}}\newcommand{\N}{\mathbb{N}}
\newcommand{\Z}{\mathbb{Z}}\newcommand{\E}{\mathbb{E}}\newcommand{\Prob}{\mathbb{P}}
\newcommand{\Hilb}{\mathcal{H}}\newcommand{\ii}{\mathrm{i}}\newcommand{\dd}{\mathrm{d}}
\newcommand{\ee}{\mathrm{e}}\newcommand{\QFT}{\mathrm{QFT}}\newcommand{\CNOT}{\mathrm{CNOT}}
\newcommand{\Uf}{U_f}\newcommand{\defeq}{:=}
\newcommand{\Xgate}{\mathsf{X}}\newcommand{\Ygate}{\mathsf{Y}}
\newcommand{\Zgate}{\mathsf{Z}}\newcommand{\Hgate}{\mathsf{H}}$

# Applications of Phase Estimation

**Learning objectives.**

-   Estimate ground-state energies with QPE.

-   Set up amplitude estimation and its quadratic advantage.

-   Formulate the eigenvalue problem and near-optimal ground-state preparation.

-   Cast order finding---the quantum core of Shor's factoring algorithm---as
    phase estimation of a modular-multiplication unitary, and recover the
    order by continued fractions.

Phase estimation (that chapter) is not an end in itself but a workhorse
subroutine. This lecture assembles its three most consequential applications in
scientific computing. First, we apply QPE to the unitary $\ee^{-\ii Ht}$ to read
off the eigenphases---and hence the energies---of a Hamiltonian $H$, obtaining the
*ground-state energy estimation* algorithm whose cost is governed by the
overlap between the input state and the true ground state. Second, we show that
*amplitude estimation*---the task of learning the weight $\sin^2(\theta/2)$
of a "good" component in a prepared state---is nothing but phase estimation of
the Grover iterate of that chapter, and that this reduction turns the classical
$\bigo(1/\epsilon^2)$ Monte Carlo cost into a quadratically better
$\bigo(1/\epsilon)$. Third, we move from estimating an eigenvalue to
*preparing* the corresponding eigenstate, describing at the statement level
how a spectral filter---built from the quantum singular value transformation
(QSVT) of that chapter---prepares the ground state with a cost that scales as
the inverse square root of the overlap and the inverse of the spectral gap.
Finally, we close with the *lower bounds* that show these algorithms are
essentially optimal: a metrological Heisenberg limit for phase estimation, a
Grover-type $\Omega(1/\sqrt{p_0})$ and $\Omega(1/\Delta)$ bound for ground-state
preparation, and a pointer to the no-fast-forwarding theorem that will recur in
Part III.

Throughout, $H$ denotes a Hermitian matrix on $\C^N$ with $N=2^n$ and
eigendecomposition
$$H\ket{\psi_j}=\lambda_j\ket{\psi_j},
  \qquad \lambda_0\le\lambda_1\le\cdots\le\lambda_{N-1},$$
so that $\lambda_0$ is the *ground-state energy* and $\ket{\psi_0}$ the
*ground state*. The *spectral gap* is $\Delta\defeq\lambda_1-\lambda_0$.
We follow the QPE conventions of that chapter: eigenphases live in $[0,1)$, a
$t$-ancilla register resolves them to $T=2^t$ levels, and $\tilde\varphi_{k}=k/T$
denotes a candidate estimate.

## Ground-state energy estimation

The most direct application of QPE to physics is to estimate the lowest energy of
a Hamiltonian. Two Hamiltonians recur as running examples.

**Example (Transverse-field Ising model).**

The one-dimensional transverse-field Ising model on $n$ sites with nearest-neighbor
coupling is
$$H=-\sum_{i=1}^{n-1} Z_i Z_{i+1}-g\sum_{i=1}^{n} X_i,$$
where $X_i,Z_i$ are Pauli operators acting on site $i$ and $g$ tunes the field.
The matrix $H$ has dimension $2^n$, exponential in the number of sites, so
classical diagonalization is intractable for large $n$.

**Example (Fermionic Hamiltonian in second quantization).**

The central Hamiltonian of quantum chemistry describes electrons occupying a fixed
set of $n$ single-particle *spin-orbitals* $\{\chi_1,\dots,\chi_n\}$. After the
Born--Oppenheimer approximation freezes the nuclei, the electronic energy is
governed by the *second-quantized* Hamiltonian
$$H=\sum_{i,j=1}^{n} T_{ij}\,\hat a_i\dg\hat a_j
   +\sum_{i,j,k,l=1}^{n} V_{ijkl}\,\hat a_i\dg\hat a_j\dg\hat a_l\hat a_k,$$
whose ingredients we now unpack.

*Occupation-number states.* Instead of tracking electron *positions*,
second quantization tracks which spin-orbitals are *occupied*. A basis state is
a bit string $\ket{f_1 f_2\cdots f_n}$ with $f_i\in\{0,1\}$ recording whether orbital
$i$ is empty or filled; these span the $2^{n}$-dimensional Fock space. The Pauli
exclusion principle---at most one electron per spin-orbital---is automatic, since
$f_i\le1$.

*Creation and annihilation operators.* The operator $\hat a_i\dg$ *adds* an
electron to orbital $i$ and $\hat a_i$ *removes* one,
$$\hat a_i\dg\ket{\dots f_i{=}0\dots}=(-1)^{s_i}\ket{\dots f_i{=}1\dots},\qquad
  \hat a_i\ket{\dots f_i{=}1\dots}=(-1)^{s_i}\ket{\dots f_i{=}0\dots},$$
while $\hat a_i\dg$ annihilates an already-filled orbital and $\hat a_i$ an empty one
($\hat a_i\dg\ket{f_i{=}1}=0$, $\hat a_i\ket{f_i{=}0}=0$). Because electrons are
*fermions*, the wavefunction must change sign under exchange; this is carried by
the factor $(-1)^{s_i}$ with $s_i=\sum_{p<i}f_p$ counting the electrons in earlier
orbitals. Equivalently, the operators obey the *canonical anticommutation
relations*
$$\{\hat a_i,\hat a_j\dg\}=\delta_{ij}\,\id,\qquad
  \{\hat a_i,\hat a_j\}=\{\hat a_i\dg,\hat a_j\dg\}=0,
  \qquad \{A,B\}\defeq AB+BA.$$
It is the *anti*commutator---not the commutator of bosons---that is the
algebraic fingerprint of fermionic statistics.

*The two terms.* The coefficients are molecular integrals precomputed classically
from the chosen orbitals (in atomic units). The *one-electron integrals*
$$T_{ij}=\int\chi_i^{*}(\mathbf r)\Bigl(-\tfrac12\nabla^{2}+V_{\mathrm{nuc}}(\mathbf r)\Bigr)
          \chi_j(\mathbf r)\,d\mathbf r$$
hold the kinetic energy and electron--nucleus attraction, and $\hat a_i\dg\hat a_j$
moves one electron from orbital $j$ to orbital $i$. The *two-electron integrals*
$V_{ijkl}=\iint\chi_i^{*}(\mathbf r_1)\chi_j^{*}(\mathbf r_2)\,r_{12}^{-1}\,
\chi_k(\mathbf r_1)\chi_l(\mathbf r_2)\,d\mathbf r_1 d\mathbf r_2$ encode the Coulomb
repulsion between electrons, and $\hat a_i\dg\hat a_j\dg\hat a_l\hat a_k$ scatters a
pair of them. The lowest eigenvalue $\lambda_0$ of $H$ is the molecular ground-state
energy sought by electronic-structure theory.

*The Jordan--Wigner transform.* To run on a quantum computer we identify each
spin-orbital with one qubit---occupied $\leftrightarrow\ket1$, empty
$\leftrightarrow\ket0$---so a Fock basis state *is* a computational basis state.
The single-qubit operators
$$\tfrac12(X+\ii Y)=\ket0\!\bra1 \quad(\text{lowering, ``annihilate''}),\qquad
  \tfrac12(X-\ii Y)=\ket1\!\bra0 \quad(\text{raising, ``create''})$$
flip one qubit between filled and empty. On their own, however, operators on
different sites *commute*, whereas fermionic operators must *anticommute*.
The Jordan--Wigner transform fixes this by attaching a string of $Z$'s,
$$\hat a_i= Z^{\otimes(i-1)}\otimes\tfrac12(X+\ii Y)\otimes\id^{\otimes(n-i)},
  \qquad
  \hat a_i\dg= Z^{\otimes(i-1)}\otimes\tfrac12(X-\ii Y)\otimes\id^{\otimes(n-i)}.$$
The leading $Z^{\otimes(i-1)}$ evaluates the parity $(-1)^{s_i}$ of the orbitals
*below* $i$ (recall $Z\ket1=-\ket1$, $Z\ket0=\ket0$), supplying precisely the
fermionic sign; and because $Z$ anticommutes with $X\pm\ii Y$ on any shared qubit,
two such operators on different modes acquire the required extra minus sign, so the
anticommutation relations above are reproduced exactly. Substituting these
expressions turns every $\hat a_i\dg\hat a_j$ and $\hat a_i\dg\hat a_j\dg\hat a_l\hat
a_k$ into a sum of Pauli strings, yielding a qubit Hamiltonian
$H=\sum_\alpha c_\alpha P_\alpha$ of exactly the form QPE and the simulation methods
of Part III can handle.

*Cost.* The price of the $Z$ string is *locality*: a single $\hat a_i$ acts
nontrivially on up to $i$ qubits, so Jordan--Wigner Hamiltonian terms can be
$\bigo(n)$-local. Alternative encodings (Bravyi--Kitaev, parity) reduce this to
$\bigo(\log n)$-local operators at the cost of a more intricate mapping. Estimating
$\lambda_0$ of the resulting qubit Hamiltonian is the central task of quantum
chemistry on a quantum computer.

### From eigenphase to energy {#from-eigenphase-to-energy .unnumbered}

To bring QPE to bear we need a unitary whose eigenphases encode the eigenvalues of
$H$. The natural choice is the time-evolution operator. Assume, after an affine
rescaling of $H$ if necessary, that all eigenvalues lie in $(0,\tfrac12)$, and set
$$U=\ee^{\ii 2\pi H},\qquad\text{so that}\qquad
  U\ket{\psi_j}=\ee^{\ii 2\pi\lambda_j}\ket{\psi_j}.$$
Then the eigenphase of $U$ on $\ket{\psi_j}$ is exactly $\lambda_j$, and running
QPE on $U$ estimates $\lambda_j$. Producing $U$ from a description of $H$ is the
*Hamiltonian simulation* problem, the subject of that chapter; for now we
assume $U$ is available exactly and count queries to it.

The catch is that we cannot prepare the exact ground state $\ket{\psi_0}$---if we
could, the problem would be largely solved. Instead we are given an
*approximate* initial state,
$$\ket{\phi}=\sum_{k=0}^{N-1} c_k\ket{\psi_k},
  \qquad p_0\defeq\abs{\braket{\phi|\psi_0}}^{2}=\abs{c_0}^2,$$
whose *overlap* $p_0$ with the ground state is bounded below but generally
less than $1$. By linearity, QPE acts on each eigencomponent of $\ket{\phi}$
independently (that remark): a single run returns an estimate of the
eigenphase $\lambda_k$ with probability $p_k=\abs{c_k}^2$, collapsing the system
to $\ket{\psi_k}$. In particular the ground-state energy $\lambda_0$ is returned
with probability $p_0$ per run; the full circuit is that figure.

<figure>
<p><em>[quantum circuit — see the PDF]</em></p>
<figcaption>Ground-state energy estimation: phase estimation on <span class="math inline"><em>U</em> = ⅇ<sup>ⅈ2<em>π</em><em>H</em></sup></span>
(that equation), fed the trial state <span class="math inline">$\ket{\phi}$</span> of ground-state overlap <span class="math inline"><em>p</em><sub>0</sub></span>
(that equation). The controlled powers <span class="math inline"><em>U</em><sup>2<sup><em>k</em></sup></sup></span> are controlled Hamiltonian
simulations; by linearity a single run returns an estimate of some eigen-energy
<span class="math inline"><em>λ</em><sub><em>k</em></sub></span> with probability <span class="math inline">$\abs{c_k}^2$</span>, so the ground energy <span class="math inline"><em>λ</em><sub>0</sub></span> appears with
probability <span class="math inline"><em>p</em><sub>0</sub></span>. The system register (one wire here) carries all <span class="math inline"><em>n</em></span> system qubits.</figcaption>
</figure>

### The min-over-runs estimator {#the-min-over-runs-estimator .unnumbered}

Because $\lambda_0$ is the *smallest* energy, a natural strategy is:

**Definition (Ground-state energy estimation by repeated QPE).**

Run QPE on $U=\ee^{\ii 2\pi H}$ with input $\ket{\phi}$ a total of $M$ times,
recording the phase estimates $\tilde\varphi^{(1)},\dots,\tilde\varphi^{(M)}$.
Return the minimum $\min_\ell\tilde\varphi^{(\ell)}$ as the estimate of
$\lambda_0$.

If every $\lambda_k$ had an exact $t$-bit representation, one run would return
$\lambda_0$ exactly with probability $p_0$, and $M=\bigo(p_0^{-1}\log\delta^{-1})$
runs would suffice to see it at least once with probability $1-\delta$. The
subtlety is the *aliasing effect*: when $\lambda_k$ is not a dyadic rational,
the geometric-sum tails of that section put nonzero weight on outcomes
$\tilde\varphi_{k'}<\lambda_0$, i.e. QPE can report an energy *below* the
true ground-state energy. Taking the minimum over runs makes such underestimates
dangerous, and controlling them is what drives the cost.

**Theorem (Overlap-dependent cost of ground-state energy estimation).**

Let $H$ have all eigenvalues in $(\epsilon,\tfrac12)$ and let $p_0$ be the initial
overlap in the equation above. The estimator of that definition
returns $\hat\lambda_0$ with $\abs{\hat\lambda_0-\lambda_0}\le\epsilon$ with
probability at least $1-\delta$ provided one uses
$$t=d+\bigo\bigl(\log p_0^{-1}+\log\delta^{-1}+\log\log\delta^{-1}\bigr)$$
ancilla qubits (with $\epsilon=2^{-d}$), circuit depth
$T=\bigo\bigl((\epsilon p_0\delta)^{-1}\log\delta^{-1}\bigr)$, and
$M=\bigo(p_0^{-1}\log\delta^{-1})$ repetitions. The total query cost is
$$\bigo\bigl(\epsilon^{-1}\delta^{-1}p_0^{-2}(\log\delta^{-1})^{2}\bigr).$$

*Proof (Proof sketch).*

Write $T=2^{t}$ and let $\tilde\varphi_{k'}=k'/T$ be the measurement grid. By the
Fejér-kernel analysis of that section, the probability of outcome $k'$ on
a single run is $\sum_k\abs{c_k}^2 F_T(\lambda_k-\tilde\varphi_{k'})$ with a
kernel obeying $F_T(x)\le 1/(4T^2\abs{x}_1^2)$, where $\abs{\cdot}_1$ is the
wrap-around distance. *Underestimation:* when
$\tilde\varphi_{k'}\le\lambda_0-\epsilon$ we have
$\abs{\lambda_k-\tilde\varphi_{k'}}_1\ge\lambda_0-\tilde\varphi_{k'}\ge\epsilon$
for every $k$, so summing the tail gives
$$\Prob(\tilde\varphi_{k'}\le\lambda_0-\epsilon)
  \le\frac{1}{4T\epsilon}+\frac{1}{4(T\epsilon)^{2}}\le\frac{\delta'}{2},$$
on setting $T\epsilon=\delta'^{-1}$. Over $M$ runs the minimum underestimates with
probability at most $M\delta'/2$, so choosing $\delta'=M^{-1}\delta$ keeps it
below $\delta/2$. *Overestimation:* conditioned on hitting $\ket{\psi_0}$
(probability $p_0$), the same kernel bound gives
$\Prob(\abs{\tilde\varphi_{k'}-\lambda_0}_1<\epsilon)\ge p_0(1-\delta')$, so a
single run overshoots by $\epsilon$ with probability at most $1-p_0/2$; over $M$
runs the minimum overshoots with probability
$\le(1-p_0/2)^M\le\ee^{-p_0 M/2}$, which is $\le\delta/2$ once
$M=\lceil 2p_0^{-1}\log(2/\delta)\rceil$. Combining $\delta'=M^{-1}\delta$ with
$T\epsilon=\delta'^{-1}$ yields the stated qubit count, depth, and---multiplying
depth by repetitions---the total cost.

**Remark (The overlap $p_0$ is the bottleneck).**

that theorem exposes a delicate interplay. Suppressing overshoot forces
$M=\bigo(p_0^{-1})$ repetitions; but more repetitions raise the chance that
*some* run undershoots, which forces $\delta'=\bigo(M^{-1})=\bigo(p_0)$ and
hence circuit depth $T=\bigo(\delta'^{-1})=\bigo(p_0^{-1})$. The product is
$TM=\bigo(p_0^{-2})$: plain QPE has a *quadratic* dependence on the inverse
overlap. When $p_0$ is exponentially small---as it can be for large chemical
systems---this is prohibitive. that section shows how spectral filtering
improves the dependence to $\bigo(p_0^{-1/2})$.

**Remark (Energy estimation does not need a gap).**

Estimating $\lambda_0$ does not require a positive spectral gap $\Delta$: any
estimate $\lambda$ with $\abs{\lambda-\lambda_0}<\epsilon$ is a good energy
estimate even if it happens to come from an excited state clustered near the
ground state (Weyl's inequality guarantees eigenvalue stability). Preparing the
ground *state*, by contrast, becomes ill-posed as $\Delta\to0$: the ground
and first-excited eigenvectors are then nearly degenerate and cannot be separated
by any filter (the Davis--Kahan $\sin\theta$ theorem). This distinction organizes
that section.

## Amplitude estimation

Grover's algorithm (that chapter) rotates a state toward a marked "good"
subspace; amplitude *estimation* instead *measures* the weight of that
subspace, and it does so by feeding the Grover iterate into phase estimation. The
result is a quadratic speedup over classical sampling for a huge range of
estimation tasks.

**Definition (Amplitude estimation).**

Let $U_{\phi}$ prepare a state $U_{\phi}\ket{0^n}=\ket{\phi_0}$ that decomposes as
$$\ket{\phi_0}=\sqrt{P}\,\ket{\phi_{\mathrm{good}}}
              +\sqrt{1-P}\,\ket{\phi_{\mathrm{bad}}},
  \qquad \sqrt{P}=\sin\frac{\theta}{2},$$
where $\ket{\phi_{\mathrm{good}}}\perp\ket{\phi_{\mathrm{bad}}}$. The
*amplitude-estimation problem* is to estimate $P$ (equivalently the angle
$\theta$) to additive precision $\epsilon$.

Classically, if one can only sample---prepare $\ket{\phi_0}$, measure, and check
whether the outcome is good---then $P$ is a Bernoulli mean, and by the variance of
a Bernoulli estimator one needs $\bigo(1/\epsilon^2)$ samples to resolve $P$ to
additive error $\epsilon$. The quantum algorithm does better by working
coherently.

**Theorem (Amplitude estimation as phase estimation of the Grover iterate).**

Let $R_{\mathrm{good}}=\id-2\proj{\phi_{\mathrm{good}}}$ reflect about the good
subspace and $R_{\phi_0}=\id-2\proj{\phi_0}$ reflect about the initial state, and
let $G=R_{\phi_0}R_{\mathrm{good}}$ be the Grover iterate. On the invariant plane
$\mathcal P=\operatorname{span}\{\ket{\phi_{\mathrm{good}}},
\ket{\phi_{\mathrm{bad}}}\}$, $G$ acts as rotation by $\theta$ and has eigenstates
$$\ket{\phi_{\pm}}=\tfrac{1}{\sqrt2}\bigl(\ket{\phi_{\mathrm{good}}}
                    \pm\ii\,\ket{\phi_{\mathrm{bad}}}\bigr),
  \qquad G\ket{\phi_{\pm}}=\ee^{\pm\ii\theta}\ket{\phi_{\pm}}.$$
The prepared state has overlap $\abs{\braket{\phi_0|\phi_{\pm}}}^2=\tfrac12$ with
each eigenstate, so phase estimation of $G$ on input $\ket{\phi_0}$ returns an
estimate of $+\theta$ or $-\theta$, each with probability $\tfrac12$, and no
post-selection is needed.

*Proof.*

By the amplitude-amplification analysis of that section, $G$ restricted to
$\mathcal P$ is the rotation
$\left(\begin{smallmatrix}\cos\theta&\sin\theta\\-\sin\theta&\cos\theta
\end{smallmatrix}\right)$ in the basis
$\{\ket{\phi_{\mathrm{good}}},\ket{\phi_{\mathrm{bad}}}\}$. A real rotation by
$\theta$ has eigenvalues $\ee^{\pm\ii\theta}$ with eigenvectors
$\tfrac{1}{\sqrt2}(\ket{\phi_{\mathrm{good}}}\pm\ii\ket{\phi_{\mathrm{bad}}})
=\ket{\phi_{\pm}}$, as stated. Writing
$\ket{\phi_0}=\sin(\theta/2)\ket{\phi_{\mathrm{good}}}
+\cos(\theta/2)\ket{\phi_{\mathrm{bad}}}$ and projecting,
$$\abs{\braket{\phi_{\pm}|\phi_0}}^2
  =\tfrac12\abs{\sin\tfrac{\theta}{2}\mp\ii\cos\tfrac{\theta}{2}}^2
  =\tfrac12\bigl(\sin^2\tfrac{\theta}{2}+\cos^2\tfrac{\theta}{2}\bigr)
  =\tfrac12.$$
Thus $\ket{\phi_0}=\tfrac{1}{\sqrt2}(\ee^{\ii\alpha_+}\ket{\phi_+}
+\ee^{\ii\alpha_-}\ket{\phi_-})$ is an equal superposition of the two
eigenstates; QPE (that definition) applied to $G$ collapses onto
$\ket{\phi_\pm}$ with probability $\tfrac12$ each and returns an estimate of the
corresponding eigenphase $\pm\theta$. Since $P=\sin^2(\theta/2)$ is even in
$\theta$, either outcome determines $P$.

The resulting circuit---prepare $\ket{\phi_0}=U_{\phi}\ket{0^n}$, then phase-estimate the
Grover iterate $G$---is shown in that figure.

<figure>
<p><em>[quantum circuit — see the PDF]</em></p>
<figcaption>Amplitude estimation: prepare <span class="math inline">$\ket{\phi_0}=U_{\phi}\ket{0^n}$</span>, then run phase
estimation on the Grover iterate <span class="math inline"><em>G</em> = <em>R</em><sub><em>ϕ</em><sub>0</sub></sub><em>R</em><sub>good</sub></span>, whose eigenvalues on the
invariant plane are <span class="math inline">ⅇ<sup>±ⅈ<em>θ</em></sup></span> (that theorem). The measured <span class="math inline"><em>t</em></span>-bit
estimate of <span class="math inline"><em>θ</em></span> yields <span class="math inline"><em>P</em> = sin<sup>2</sup>(<em>θ</em>/2)</span> to additive error <span class="math inline">$\bigo(2^{-t})$</span>, i.e. <span class="math inline">$\bigo(1/\epsilon)$</span> queries to <span class="math inline"><em>G</em></span>—quadratically fewer than the <span class="math inline">$\bigo(1/\epsilon^2)$</span>
samples of classical Monte Carlo.</figcaption>
</figure>

### Query count and the quadratic advantage {#query-count-and-the-quadratic-advantage .unnumbered}

Let QPE use $t=d+\lceil\log\delta^{-1}\rceil$ ancilla qubits, so it estimates
$\theta$ to precision $\epsilon'=2^{-d}$ with success probability $1-\delta$
(that theorem). The relationship $P=\sin^2(\theta/2)$ converts an
error in $\theta$ into an error in $P$.

**Theorem (Amplitude estimation error and cost).**

With $\hat P=\sin^2(\tilde\theta/2)$ the estimator built from a QPE estimate
$\tilde\theta$ of $\theta$ with $\abs{\tilde\theta-\theta}\le\epsilon'$,
$$\abs{\hat P-P}\le\sqrt{P(1-P)}\,\epsilon'+\frac{\epsilon'^2}{4}.$$
Consequently, if $P$ is bounded away from $0$ and $1$ one takes
$\epsilon'=\bigo(\epsilon)$ and QPE of $G$ costs $\bigo(1/\epsilon)$ queries to
$G$ (hence to $U_\phi$); if $P$ is small one estimates it to multiplicative error
$\epsilon$ by taking $\epsilon'=\sqrt{P}\,\epsilon$, at cost
$\bigo(P^{-1/2}\epsilon^{-1})$. In both cases this is a quadratic improvement over
the $\bigo(\epsilon^{-2})$ and $\bigo(P^{-1}\epsilon^{-2})$ sample counts of
Monte Carlo estimation.

*Proof.*

Using the product-to-sum identities,
$$\sin^2\tfrac{\tilde\theta}{2}-\sin^2\tfrac{\theta}{2}
   =\sin\tfrac{\theta}{2}\cos\tfrac{\theta}{2}\sin(\tilde\theta-\theta)
    +\bigl(1-2\sin^2\tfrac{\theta}{2}\bigr)\sin^2\tfrac{\tilde\theta-\theta}{2}.$$
The first term has magnitude
$\sqrt{P(1-P)}\,\abs{\sin(\tilde\theta-\theta)}\le\sqrt{P(1-P)}\,\epsilon'$ since
$\sin(\theta/2)\cos(\theta/2)=\tfrac12\sin\theta=\sqrt{P(1-P)}$ and
$\abs{\sin x}\le\abs{x}$; the second is bounded by
$\abs{1-2P}\sin^2(\epsilon'/2)\le\epsilon'^2/4$. Adding gives
the equation above. When $P(1-P)=\Theta(1)$ the linear term dominates and
$\epsilon'=\bigo(\epsilon)$ suffices, so QPE queries $G$ a total of
$\bigo(1/\epsilon')=\bigo(1/\epsilon)$ times. When $P$ is small,
$\abs{\hat P-P}\approx\sqrt P\,\epsilon'$, and demanding this be $<P\epsilon$
forces $\epsilon'=\sqrt P\,\epsilon$ and cost
$\bigo(\epsilon'^{-1})=\bigo(P^{-1/2}\epsilon^{-1})$. The classical Monte Carlo
counts follow from the Bernoulli variance $P(1-P)$.

**Example (Accelerating the Hadamard test).**

Recall the Hadamard test of that section, which estimates
$\Re\braket{\psi|U|\psi}$ through the outcome probability
$p(0)=\tfrac12(1+\Re\braket{\psi|U|\psi})$. Let $\ket{\psi}=U_\psi\ket{0^n}$. The
combined circuit that applies $H$ to an ancilla, controlled-$U_\psi$ then
controlled-$U$, produces
$$\ket{\phi_0}=\tfrac12\ket{0}\bigl(\ket{\psi}+U\ket{\psi}\bigr)
              +\tfrac12\ket{1}\bigl(\ket{\psi}-U\ket{\psi}\bigr)
             =\sqrt{P}\,\ket{\phi_{\mathrm{good}}}
              +\sqrt{1-P}\,\ket{\phi_{\mathrm{bad}}},$$
with the good subspace flagged by the ancilla $\ket{0}$, so
$P=p(0)$. Here $R_{\mathrm{good}}=(\id-2\proj{0})\otimes\id^{\otimes n}
=-Z\otimes\id^{\otimes n}$ is a single-qubit reflection and $R_{\phi_0}$ is built
from the state-preparation unitary. Running amplitude estimation on
$G=R_{\phi_0}R_{\mathrm{good}}$ estimates $p(0)$---and hence
$\Re\braket{\psi|U|\psi}$---to precision $\epsilon$ using $\bigo(1/\epsilon)$
queries, rather than the $\bigo(1/\epsilon^2)$ repetitions of naive sampling.

**Remark (In what sense is amplitude estimation "Heisenberg-limited"?).**

Amplitude estimation is often called Heisenberg-limited because it reduces to
phase estimation of the Grover eigenphase and achieves error $\bigo(1/\mathcal T)$
in the "Grover time" $\mathcal T=\bigo(2^{t})$---the same $1/\mathcal T$ scaling
as the metrological Heisenberg limit of that section. The interpretation
differs, though: the metrological limit concerns a phase generated by physical
time evolution $\ee^{-\ii Ht}$, whereas here $G$ is a *constructed* unitary
and $\mathcal T$ counts applications of the state-preparation oracle.

## Eigenvalue problems and ground-state preparation

We now move from estimating $\lambda_0$ to *preparing* $\ket{\psi_0}$. As
noted in that remark, this requires a positive spectral gap $\Delta$.
The QPE-based approach of that section already prepares $\ket{\psi_0}$ as a
by-product---the system collapses to $\ket{\psi_k}$ when the ancilla reads
$\lambda_k$---but at cost $\bigo(p_0^{-2})$ in the inverse overlap. The modern
approach replaces the slowly-decaying Fejér kernel of QPE by a *sharp
spectral filter* implemented with QSVT (that chapter), improving the overlap
dependence to $\bigo(p_0^{-1/2})$.

### The filtering idea {#the-filtering-idea .unnumbered}

Suppose we know a threshold $\mu$ separating the ground energy from the rest,
$\lambda_0\le\mu-\tfrac{\Delta}{2}<\mu+\tfrac{\Delta}{2}\le\lambda_1$. Consider
the (shifted) step function
$$\theta_\mu(x)=\begin{cases}1,& x<\mu,\\ 0,& x>\mu,\end{cases}$$
so that the matrix function $\theta_\mu(H)=\proj{\psi_0}$ is exactly the projector
onto the ground state. If we could apply $\theta_\mu(H)$ to the initial state
$\ket{\phi}$, we would obtain $\theta_\mu(H)\ket{\phi}=c_0\ket{\psi_0}$, which
after normalization is precisely $\ket{\psi_0}$. QSVT lets us apply not the exact
step function---which is discontinuous and would need infinite-degree
polynomials---but a polynomial $F$ that approximates it away from the forbidden
window $(\mu-\tfrac{\Delta}{2},\mu+\tfrac{\Delta}{2})$.

**Definition (Spectral filter polynomial).**

A real even polynomial $F$ is an $(\epsilon',\mu,\Delta)$-*ground-state
filter* if
$$\abs{F(x)-1}\le\epsilon'\ \ \forall x\in[0,\mu-\tfrac{\Delta}{2}],
  \qquad
  \abs{F(x)}\le\epsilon'\ \ \forall x\in[\mu+\tfrac{\Delta}{2},1].$$
Such an $F$ can be built from a polynomial approximation to the sign function,
with degree $\deg F=\bigo(\Delta^{-1}\log(1/\epsilon'))$.

The degree bound is the crux: resolving a gap $\Delta$ costs $\bigo(1/\Delta)$ in
polynomial degree, and hence in the number of block-encoding queries, while the
precision $\epsilon'$ enters only logarithmically. Applying $F(H)$ to
$\ket{\phi}=\sum_k c_k\ket{\psi_k}$,
$$F(H)\ket{\phi}=\sum_k c_k F(\lambda_k)\ket{\psi_k}
   =\underbrace{c_0 F(\lambda_0)}_{\approx c_0}\ket{\psi_0}
    +\sum_{k\ge1}\underbrace{c_k F(\lambda_k)}_{\bigo(\epsilon')}\ket{\psi_k},$$
so the excited components are suppressed to $\bigo(\epsilon')$ while the ground
component is preserved. The QSVT circuit realizes $F(H)$ as a block encoding: on
input $\ket{0^m}\ket{\phi}$ it outputs $\ket{0^m}F(H)\ket{\phi}+\ket{\perp}$, and
measuring the ancilla flag $\ket{0^m}$ succeeds with probability
$\approx p_0$, leaving the system in a state of fidelity $1-\bigo(\epsilon'^2/p_0)$
with $\ket{\psi_0}$.

### Near-optimal cost {#near-optimal-cost .unnumbered}

Naively repeating until the flag reads $\ket{0^m}$ costs $\bigo(p_0^{-1})$ trials;
amplitude amplification (that section) boosts the success amplitude and
reduces this to $\bigo(p_0^{-1/2})$. This is the near-optimal ground-state
preparation result.

**Theorem (Ground-state preparation with a known energy window).**

Let $U_H$ be an $(\alpha,m)$-block encoding of $H$ with spectral gap
$\Delta=\lambda_1-\lambda_0>0$, let $\ket{\phi}=U_I\ket{0^n}$ have overlap
$p_0=\abs{\braket{\phi|\psi_0}}^2>0$, and let $\mu$ satisfy
$\lambda_0\le\mu-\tfrac{\Delta}{2}<\mu+\tfrac{\Delta}{2}\le\lambda_1$. Then the
ground state $\ket{\psi_0}$ can be prepared to fidelity at least $1-\epsilon$ using
$$\bigo\bigl(\alpha\,\Delta^{-1}\,p_0^{-1/2}\,\log(p_0^{-1}\epsilon^{-1})\bigr)$$
queries to $U_H$ and its inverse, and $\bigo(p_0^{-1/2})$ queries to $U_I$ and its
inverse.

*Proof (Proof idea).*

Take $F$ an $(\epsilon',\mu,\Delta)$-filter of degree
$\bigo(\Delta^{-1}\log(1/\epsilon'))$ (that definition) and implement
$F(H)$ by QSVT, which uses $\bigo(\deg F)$ queries to $U_H,U_H\dg$ per application.
The post-selected state has fidelity $1-\bigo(\epsilon'^2/p_0)$ with
$\ket{\psi_0}$, so $\epsilon'=\Theta(\sqrt{\epsilon p_0})$ gives fidelity
$1-\epsilon$ and makes $\log(1/\epsilon')=\bigo(\log(p_0^{-1}\epsilon^{-1}))$.
Amplitude amplification on the flag $\ket{0^m}$, whose success amplitude is
$\Theta(\sqrt{p_0})$, needs $\bigo(p_0^{-1/2})$ rounds; multiplying the per-round
query depth $\bigo(\alpha\Delta^{-1}\log(1/\epsilon'))$ by these rounds gives the
stated $U_H$-query count, and each round uses $\bigo(1)$ calls to $U_I$.

**Remark (Removing the a priori energy estimate).**

that theorem assumes a known window $\mu$. In practice $\mu$ is unknown and
must itself be computed. A binary search over $\mu$, in which each step solves a
*fuzzy bisection* decision ("is $\lambda_0$ below $\mu-h$ or above
$\mu+h$?") by estimating $\norm{F(H)\ket{\phi}}$ with amplitude estimation,
locates $\lambda_0$ to precision $\epsilon$ using
$\widetilde{\bigo}(\alpha\,\epsilon^{-1}p_0^{-1/2}\log\delta^{-1})$ queries to
$U_H$. The precision $\epsilon$ plays the role of an effective gap, so this
achieves Heisenberg-limited $\bigo(1/\epsilon)$ scaling in the energy precision
*without* assuming a positive gap; when a gap $\Delta$ is known one runs it
only to precision $\Delta/2$ and then invokes that theorem.

**Remark (Why $p_0^{-1/2}$ and not $p_0^{-1}$).**

The square-root improvement over plain QPE (that remark) is exactly
the amplitude-amplification gain of that section: preparing the ground state
is "finding a marked item" (the $\ket{\psi_0}$ component of $\ket{\phi}$, of
weight $p_0$), and amplitude amplification finds it in $\bigo(p_0^{-1/2})$ rounds
rather than the $\bigo(p_0^{-1})$ of incoherent sampling. The sharp QSVT filter is
what makes the "marking" clean enough for amplification to apply.

## Lower bounds

Are the algorithms above wasteful, or essentially optimal? Three lower bounds---one
metrological, two from unstructured search---show that their key resource
dependences cannot be improved.

### The Heisenberg limit for phase estimation {#the-heisenberg-limit-for-phase-estimation .unnumbered}

For estimating an eigenphase $\varphi$ through queries to $U(t)=\ee^{-\ii Ht}$, the
natural resource is the total evolution time
$\mathcal T=\sum_p\abs{t_p}$. As discussed in that section, coherent phase
accumulation attains precision $\epsilon\sim1/\mathcal T$ (the *Heisenberg
limit*), quadratically better than the $\epsilon\sim1/\sqrt{\mathcal T}$
shot-noise scaling of incoherent repetition. This is not merely achievable but
optimal.

**Theorem (Heisenberg-limit lower bound).**

Consider any protocol that applies controlled evolutions $U(t_1),\dots,U(t_N)$ to
a fixed eigenstate $\ket{\psi}$ with $U(t)\ket{\psi}=\ee^{\ii t\varphi}\ket{\psi}$,
interleaved with arbitrary unitaries acting only on the ancilla register, and
produces a locally unbiased estimator $\hat\varphi$ of $\varphi$. Then, with total
evolution time $\mathcal T=\sum_p\abs{t_p}$,
$$\Var(\hat\varphi)\ge\frac{1}{4\mathcal T^{2}}.$$
Hence no protocol in this model beats the Heisenberg scaling
$\epsilon=\Omega(1/\mathcal T)$.

*Proof (Proof idea).*

Each controlled query $U(t_p)$ imprints an effective phase rotation
$V_p(\varphi)=(\id-\Pi_p)+\ee^{\ii t_p\varphi}\Pi_p$ on the ancilla, where $\Pi_p$
is the projector onto the "activated" subspace, so
$\norm{\partial_\varphi V_p}=\abs{t_p}\,\norm{\Pi_p}\le\abs{t_p}$. The final state
$\ket{\Psi(\varphi)}$ is a product of the $V_p$ and ancilla unitaries; by the
triangle inequality $\norm{\partial_\varphi\Psi}\le\sum_p\abs{t_p}=\mathcal T$. For
a pure-state family the quantum Fisher information obeys
$F_Q\le4\norm{\partial_\varphi\Psi}^2\le4\mathcal T^2$, and the quantum
Cramér--Rao bound (that chapter) gives
$\Var(\hat\varphi)\ge1/F_Q\ge1/(4\mathcal T^2)$.

The Fisher-information viewpoint also explains why the *spacing* of evolution
times matters: only exponentially growing times $t_p\propto2^{p}$---as in Kitaev's
method and standard QPE---accumulate the quadratic Fisher information
$\sum_p t_p^2=\Theta(\mathcal T^2)$ needed to saturate
that theorem. A uniform grid $t_p=p\tau$ gives only
$\sum_p t_p^2=\Theta(\mathcal T^{2}/N)$ and reaches precision
$\epsilon=\Theta(\mathcal T^{-2/3})$, between shot noise and the Heisenberg limit.

### Search-based bounds for ground-state preparation {#search-based-bounds-for-ground-state-preparation .unnumbered}

The cost $\bigo(p_0^{-1/2}\Delta^{-1})$ of that theorem is optimal in
*both* parameters, as one sees by reducing unstructured search---which needs
$\Omega(\sqrt N)$ queries (that chapter)---to ground-state preparation.

**Theorem (Overlap lower bound).**

There is a family of block-encoded Hamiltonians with constant gap
$\Delta=\Omega(1)$ and overlap $p_0\to0$ for which preparing $\ket{\psi_0}$ to
fidelity at least $\sqrt3/2$ requires $\Omega(p_0^{-1/2})$ queries to $U_H$.

*Proof (Proof idea).*

Take $U_H=U_t=\id-2\proj{t}$ for an unknown marked string $t$; it is Hermitian and
unitary, is its own $(1,0)$-block encoding, has $\ket{t}$ as its unique ground
state with gap $2$, and the uniform superposition $\ket{u}=H^{\otimes n}\ket{0^n}$
has overlap $\abs{\braket{u|t}}^2=1/N$, so $p_0=1/N$ and $\sqrt{p_0}=1/\sqrt N$.
Preparing the ground state to fidelity $\ge\sqrt3/2$ and measuring returns $t$ with
probability $\ge3/4$. If this used $o(p_0^{-1/2})=o(\sqrt N)$ queries it would
solve unstructured search in $o(\sqrt N)$ queries, contradicting Grover's lower
bound.

**Theorem (Gap lower bound).**

There is a family with constant overlap $p_0=\Omega(1)$ and gap $\Delta\to0$ for
which preparing $\ket{\psi_0}$ to fidelity at least $\sqrt3/2$ requires
$\Omega(\Delta^{-1})$ queries to $U_H$.

*Proof (Proof idea).*

With $U_t=\id-2\proj{t}$ and $D=\id-2\proj{u}$ (both reflections implementable by
Hadamards and one oracle call), the interpolated Hamiltonian
$H(\tfrac12)=\tfrac12(D+U_t)$ acts nontrivially only on
$\operatorname{span}\{\ket{u},\ket{t}\}$, where it has eigenvalues
$\pm1/\sqrt N$: the gap is $\Delta=2/\sqrt N\to0$, while its ground state has
constant overlap $\tfrac12+\bigo(1/\sqrt N)$ with both $\ket{u}$ and $\ket{t}$.
Preparing that ground state and measuring yields $t$ with constant probability, so
$o(\Delta^{-1})=o(\sqrt N)$ queries would again break Grover's bound.

### No fast-forwarding, and a look ahead to Part III {#no-fast-forwarding-and-a-look-ahead-to-part-iii .unnumbered}

Both bounds above trace back to the optimality of Grover search. A related and
deeper obstruction governs the *simulation* of Hamiltonians and the solution
of linear systems: the *no-fast-forwarding theorem* states that simulating a
generic Hamiltonian for time $t$ requires $\Omega(t)$ queries in the worst case---
one cannot compress long-time dynamics into sublinear cost. This is why the
quantum linear-system solvers previewed in that chapter inherit a linear
dependence on the condition number $\kappa$ (the QLSP lower bound is
$\Omega(\kappa)$, proved by encoding Hamiltonian simulation into a linear system),
and why the Hamiltonian-simulation algorithms of that chapter aim for
cost *linear* in $t$ rather than sublinear. We will meet the no-fast-forwarding
theorem in detail when we build those algorithms in Part III; for now it stands as
the fundamental limit that phase estimation, amplitude estimation, and eigenstate
preparation all press against.

## Order finding and Shor's algorithm

The applications so far have been drawn from scientific computing. We close with
the algorithm that first made phase estimation famous: *Shor's factoring
algorithm*, whose sole quantum ingredient is *order finding*, and whose
order-finding routine is nothing but QPE (that chapter) applied to a
modular-multiplication unitary. Beyond its historical weight, it is the cleanest
illustration of the recurring pattern of this lecture---encode a number-theoretic
quantity in the eigenphase of an efficiently implementable unitary, then read that
phase out with QPE---and it is why a large fault-tolerant quantum computer would
break the RSA cryptosystem.

### The number-theory wrapper: factoring reduces to order finding {#the-number-theory-wrapper-factoring-reduces-to-order-finding .unnumbered}

Let $N=PQ$ be a product of two large primes, the modulus of an RSA public key.
Classically the best known factoring algorithms run in time
$\ee^{\bigo((\log N)^{1/3}(\log\log N)^{2/3})}$ (the number field sieve),
superpolynomial in the bit length $\log N$. Shor's insight is that factoring
reduces, in randomized polynomial classical time, to the problem of computing a
*multiplicative order*.

**Definition (Multiplicative order).**

For $a$ coprime to $N$ (that is, $\gcd(a,N)=1$), the *order* of $a$ modulo
$N$ is the least positive integer $r$ with
$$a^{r}\equiv 1 \pmod N .$$

The classical wrapper around order finding is:

1.  Pick $a\in\{2,\dots,N-1\}$ uniformly at random. Compute $g=\gcd(a,N)$ by
    Euclid's algorithm. If $g>1$ then $g$ is already a nontrivial factor of
    $N$---return it. Otherwise $\gcd(a,N)=1$ and we continue.

2.  Use the quantum order-finding routine below to compute the order $r$ of
    $a\bmod N$. *This is the only quantum step.*

3.  If $r$ is odd, or if $a^{r/2}\equiv -1\pmod N$ (equivalently
    $a^{r/2}\bmod N=N-1$), the run is unlucky; discard it and return to
    step 1 with a fresh $a$.

4.  Otherwise compute $\gcd(a^{r/2}-1,\,N)$ and $\gcd(a^{r/2}+1,\,N)$. Each is
    a nontrivial factor of $N$.

Why does step 4 work? Since $a^{r}\equiv1$, the difference of squares gives
$$\bigl(a^{r/2}-1\bigr)\bigl(a^{r/2}+1\bigr)=a^{r}-1\equiv 0 \pmod N,$$
so $N\mid(a^{r/2}-1)(a^{r/2}+1)$. Minimality of $r$ forbids
$a^{r/2}\equiv 1$ (that would make $r/2$ a smaller order), and step 3 has excluded
$a^{r/2}\equiv -1$; hence $N$ divides neither factor on the left of
the equation above alone, so its prime factors must split between the
two factors. Consequently $\gcd(a^{r/2}\mp1,N)$ is a nontrivial divisor of $N$. A
standard number-theory estimate shows that a random $a$ yields an even $r$ with
$a^{r/2}\not\equiv-1$ with probability at least $1/2$ when $N$ has at least two
distinct odd prime factors, so $\bigo(1)$ repetitions suffice on average. Every
step but order finding---random choice, $\gcd$, the parity test---is elementary
classical arithmetic.

### Order finding as phase estimation {#order-finding-as-phase-estimation .unnumbered}

Fix $a$ with $\gcd(a,N)=1$ and work on the state space $\C^{2^n}$ with
$n=\lceil\log_2 N\rceil$, whose computational basis labels the residues
$\{0,1,\dots,N-1\}$ (and unused labels $\ge N$). Define the
*modular-multiplication unitary*
$$U_a\ket{y}=\ket{ay\bmod N}\quad(0\le y<N),
  \qquad U_a\ket{y}=\ket{y}\ (y\ge N).$$
Because multiplication by $a$ permutes the residues coprime to $N$ (its inverse is
multiplication by $a^{-1}\bmod N$), $U_a$ is a permutation matrix, hence unitary,
and it is implemented with $\bigo((\log N)^2)$ elementary gates by
modular-exponentiation circuits. The point is that $U_a$ has an eigenbasis whose
eigenphases expose the order $r$.

**Lemma (Eigenphases of the modular-multiplication unitary).**

Let $r$ be the order of $a\bmod N$. For each integer $s\in\{0,1,\dots,r-1\}$ the
state
$$\ket{u_s}=\frac{1}{\sqrt r}\sum_{k=0}^{r-1}
             \ee^{-2\pi\ii\, sk/r}\,\ket{a^{k}\bmod N}$$
is an eigenstate of $U_a$ with eigenvalue $\ee^{2\pi\ii\, s/r}$:
$$U_a\ket{u_s}=\ee^{2\pi\ii\, s/r}\ket{u_s}.$$
Moreover $\dfrac{1}{\sqrt r}\sum_{s=0}^{r-1}\ket{u_s}=\ket{1}$.

*Proof.*

Applying $U_a$ shifts $k\mapsto k+1$ in the cyclic group generated by $a$:
$$U_a\ket{u_s}=\frac{1}{\sqrt r}\sum_{k=0}^{r-1}
    \ee^{-2\pi\ii sk/r}\ket{a^{k+1}\bmod N}
   =\ee^{2\pi\ii s/r}\,\frac{1}{\sqrt r}\sum_{k'=0}^{r-1}
    \ee^{-2\pi\ii sk'/r}\ket{a^{k'}\bmod N},$$
where we reindexed $k'=k+1$ and used $a^{r}\equiv1$ so that the sum is periodic
with period $r$; the reindexing multiplies each term by $\ee^{2\pi\ii s/r}$. The
final identity follows because
$\frac1r\sum_{s=0}^{r-1}\ee^{-2\pi\ii sk/r}=\delta_{k,0}$ picks out the $k=0$ term
$\ket{a^0\bmod N}=\ket{1}$.

that lemma places us exactly in the QPE setting of that chapter: the
eigenphase of $U_a$ on $\ket{u_s}$ is $\varphi_s=s/r\in[0,1)$. Running QPE on
$U_a$ with the $t$-qubit ancilla register of that chapter therefore returns a
$t$-bit estimate of some $s/r$. Two facts make this practical without knowing the
eigenstates $\ket{u_s}$:

-   *Cheap input state.* We cannot prepare an individual $\ket{u_s}$, but
    the last identity of that lemma says the trivially-prepared state
    $\ket{1}$ is the uniform superposition of all the $\ket{u_s}$. By linearity
    (as in that equation for energy estimation), QPE on input
    $\ket{1}$ returns an estimate of $s/r$ for a uniformly random
    $s\in\{0,\dots,r-1\}$.

-   *Controlled powers.* QPE needs controlled $U_a^{2^{j}}$, and
    $U_a^{2^{j}}\ket{y}=\ket{a^{2^{j}}y\bmod N}$ is again a modular
    multiplication (by the precomputable constant $a^{2^{j}}\bmod N$), so the
    whole circuit is modular exponentiation, costing
    $\bigo((\log N)^{2})$ gates per controlled power and $\bigo((\log N)^{3})$
    overall.

Assembling the two ingredients gives the *order-finding circuit* of
that figure, the quantum core of Shor's algorithm.

<figure>
<p><em>[quantum circuit — see the PDF]</em></p>
<figcaption>Order finding by phase estimation on the modular-multiplication unitary
<span class="math inline">$U_a\ket{y}=\ket{ay\bmod N}$</span> (that equation). The work register is prepared in the
trivially-available <span class="math inline">$\ket{1}=\tfrac{1}{\sqrt r}\sum_{s}\ket{u_s}$</span> (that lemma),
and each controlled <span class="math inline"><em>U</em><sub><em>a</em></sub><sup>2<sup><em>k</em></sup></sup></span> is a modular multiplication by the precomputed constant
<span class="math inline"><em>a</em><sup>2<sup><em>k</em></sup></sup> mod  <em>N</em></span> (modular exponentiation). A run returns a <span class="math inline"><em>t</em></span>-bit estimate of a random
<span class="math inline"><em>s</em>/<em>r</em></span>; continued fractions then recover the order <span class="math inline"><em>r</em></span>.</figcaption>
</figure>

### Recovering $r$ by continued fractions {#recovering-r-by-continued-fractions .unnumbered}

QPE returns a dyadic rational $\tilde\varphi=k/2^{t}\approx s/r$. With
$t=2\log_2 N+\bigo(1)$ ancilla qubits, $\tilde\varphi$ is within $1/2N^{2}$ of the
true $s/r$; since two distinct fractions with denominators below $N$ differ by
more than $1/N^{2}$, the continued-fraction expansion of $\tilde\varphi$ has a
convergent $s'/r'$ (in lowest terms) that equals $s/r$ exactly. When $\gcd(s,r)=1$
this convergent's denominator $r'$ *is* $r$; otherwise it is a divisor of
$r$, and testing $a^{r'}\equiv1$ (or a few small multiples of $r'$) recovers $r$.
Continued fractions thus convert the approximate phase readout into an exact
integer order.

**Example (Factoring $N=15$ with $a=7$).**

Take $N=15$ and, having drawn $a=7$ with $\gcd(7,15)=1$, compute the order:
$7^{1}=7$, $7^{2}=49\equiv4$, $7^{3}\equiv 7\cdot4=28\equiv13$,
$7^{4}\equiv7\cdot13=91\equiv1\pmod{15}$, so $r=4$. By that lemma the
unitary $U_7$ has eigenphases $s/4\in\{0,\tfrac14,\tfrac12,\tfrac34\}$. Order
finding uses $n=4$ qubits for the residue register (since $15=1111_2$). With even
a modest ancilla register these phases are dyadic, so QPE on input $\ket{1}$
returns one of the readouts
$$0.00_2=\tfrac{0}{4},\quad 0.01_2=\tfrac14,\quad
  0.10_2=\tfrac12,\quad 0.11_2=\tfrac34,$$
each with probability $\tfrac14$. A nonzero outcome, say $\tilde\varphi=0.01_2=1/4$,
already presents $r=4$ as its denominator; the continued-fraction step is trivial
here. (The outcome $0$ carries no information about $r$ and is simply discarded.)
Steps 3--4 of the wrapper then finish the job: $r=4$ is even and
$7^{r/2}=7^{2}=49\equiv 4\not\equiv-1\pmod{15}$, so
$$\gcd\bigl(7^{2}-1,\,15\bigr)=\gcd(48,15)=3,
  \qquad
  \gcd\bigl(7^{2}+1,\,15\bigr)=\gcd(50,15)=5,$$
recovering both prime factors, $15=3\times5$. (These match Brody's worked run of
the order-finding circuit on real hardware, where the dominant outcomes for
$a\in\{2,7,8,13\}$ are exactly $\{00,01,10,11\}\!\cdot\!\tfrac14$, giving $r=4$.)

The continued-fraction machinery earns its keep only when $r$ is not a power of
two, so that $s/r$ is a repeating binary fraction and the QPE readout is a genuine
approximation. Factoring $N=21$ with $a=2$ (order $r=6$) is the standard such
example and is developed in that exercise.

### Cost and the exponential speedup {#cost-and-the-exponential-speedup .unnumbered}

Assembling the pieces: the residue and ancilla registers use $\bigo(\log N)$
qubits; QPE issues $\bigo(\log N)$ controlled powers of $U_a$, each a modular
exponentiation of gate cost $\bigo((\log N)^{2})$ (or $\widetilde{\bigo}(\log N)$
with fast multiplication), for a total quantum cost of
$\bigo((\log N)^{3})$; and the classical wrapper---$\gcd$, continued fractions,
and $\bigo(1)$ expected retries---adds only polynomially in $\log N$. The whole
algorithm therefore runs in time polynomial in the bit length, an
*exponential* speedup over the superpolynomial classical number field sieve.
Because RSA's security rests precisely on the presumed classical hardness of
factoring $N=PQ$, a scalable fault-tolerant machine running Shor's algorithm would
recover the private key from the public modulus---this is the sense in which phase
estimation "breaks the internet." The order-finding core is a strict
generalization of the QPE reasoning we used for energies in that section and
for the Grover angle in amplitude estimation (that section): in every case a
quantity of interest is written as the eigenphase of an efficiently implementable
unitary and extracted by that chapter's phase-estimation circuit.

## Exercises {#exercises .unnumbered}

**Exercise.**

*Energy resolution versus ancilla count.* Suppose $H$ has been rescaled so
that $U=\ee^{\ii 2\pi H}$ has all eigenphases in $[0,1)$, and QPE is run on an
exact eigenstate with $t$ ancilla qubits. Show that the additive resolution in the
energy $\lambda$ is $2^{-t}$, and determine how many ancilla qubits are needed to
resolve $\lambda_0$ to $10^{-3}$ (in the rescaled units). If the physical energy
range is $[E_{\min},E_{\max}]$ with $E_{\max}-E_{\min}=W$, what is the resolution
in physical units?

**Exercise.**

*Amplitude-estimation query count.* Using that theorem, determine
the number of applications of the Grover iterate $G$ needed to estimate an
amplitude $P$ with $P(1-P)=\Theta(1)$ to additive precision $\epsilon=10^{-3}$,
and compare with the number of independent samples a classical Monte Carlo
estimator would need for the same precision. State the ratio of the two costs as a
function of $\epsilon$.

**Exercise.**

*Overlap requirement for success probability $\ge\tfrac12$.* In the
QPE-based ground-state routine of that section, a single run returns (an
estimate of) $\lambda_0$ with probability $p_0$. Show that to obtain $\lambda_0$
at least once with probability $\ge\tfrac12$ in $M$ independent runs one needs
$M\ge\log 2/\log\bigl((1-p_0)^{-1}\bigr)$, and simplify this for small $p_0$.
Conclude that $M=\bigo(p_0^{-1})$ runs suffice.

**Exercise.**

*Eigenphases of the Grover iterate.* Verify by direct matrix computation that
the rotation $\left(\begin{smallmatrix}\cos\theta&\sin\theta\\-\sin\theta&
\cos\theta\end{smallmatrix}\right)$ has eigenvalues $\ee^{\pm\ii\theta}$ with
eigenvectors $\tfrac{1}{\sqrt2}(1,\mp\ii)^{\mathsf T}$, and confirm the overlaps
$\abs{\braket{\phi_0|\phi_\pm}}^2=\tfrac12$ used in that theorem.

**Exercise.**

*Filter degree versus gap.* A ground-state filter (that definition)
has degree $\deg F=\bigo(\Delta^{-1}\log(1/\epsilon'))$. Explain why the degree
must grow at least like $\Delta^{-1}$ (hint: a degree-$D$ polynomial bounded by
$1$ on $[-1,1]$ cannot change from $\approx1$ to $\approx0$ across an interval much
shorter than $\bigo(1/D)$; recall the behavior of Chebyshev polynomials). Deduce
how the query depth of that theorem scales as $\Delta\to0$ and connect
this to the gap lower bound of that theorem.

**Exercise.**

*Quadratic versus square-root overlap dependence.* Contrast the total cost of
plain QPE ground-state energy estimation (that theorem,
$\bigo(p_0^{-2})$ ignoring logs) with the QSVT-based preparation of
that theorem ($\bigo(p_0^{-1/2})$). Explain, in one or two sentences each,
which mechanism produces each power of $p_0$, and why amplitude amplification is
responsible for the improvement (cf. that remark).

**Exercise.**

*Factoring $N=21$ with $a=2$ by continued fractions.* Verify that the order
of $2\bmod 21$ is $r=6$. With a $5$-bit ancilla register, order finding on $U_2$
(that equation) returns readouts approximating $s/6$; one such readout is
$0.00101_2=5/32$. Compute the continued-fraction expansion of $5/32$, list its
convergents, and identify the convergent whose denominator is the order $r=6$
(check $2^{6}\equiv1\pmod{21}$ and that no smaller candidate denominator works).
Then complete Shor's algorithm: confirm $r$ is even and $2^{r/2}\not\equiv-1$, and
compute $\gcd(2^{3}-1,21)$ and $\gcd(2^{3}+1,21)$ to recover the factors of $21$.

**Exercise.**

*Eigenphases of the modular-multiplication unitary.* For $N=15$ and $a=7$,
list the cycle $\{7^{k}\bmod 15:k=0,1,2,3\}$ and use it to write the four
eigenstates $\ket{u_s}$ ($s=0,1,2,3$) of $U_7$ from that lemma
explicitly. Verify directly that $U_7\ket{u_1}=\ee^{2\pi\ii/4}\ket{u_1}=\ii\ket{u_1}$,
and confirm that $\tfrac12\sum_{s=0}^{3}\ket{u_s}=\ket{1}$, so that preparing the
easy state $\ket{1}$ is equivalent to sampling a uniformly random eigenphase
$s/4$.

## Further reading {#further-reading .unnumbered}

-   Lin Lin, *qasc notes*, Ch. 4 (ground-state energy estimation,
    amplitude estimation, HHL, and the Poisson-equation example).

-   Lin Lin, *live notes*, Ch. 16 (amplitude estimation, Heisenberg
    limit, and the Cramér--Rao viewpoint).

-   Lin Lin, *live notes*, Ch. 18 (eigenvalue problems, near-optimal
    ground-state preparation, and the matching lower bounds).

-   J. Brody, *The Joy of Quantum Computing*, Princeton University
    Press, 2025, Ch. 10--11 (a hands-on account of order finding via phase
    estimation and Shor's factoring algorithm, including worked
    order-finding circuits for $a\bmod 15$ and the continued-fraction
    recovery of $r$ when factoring $21$).